# agents_4_puzzles — Megaminx neighbour-model + LLM hybrid (Google Colab, g4f configurable)

Обновлено: архив репозитория уже содержит CLI-флаги `--baseline-patch-max-iters` и `--g4f-recovery-max-iters`, поэтому гибридный запуск Megaminx может работать без ручной правки `pipeline_cli.py`.

Этот Colab notebook запускает **интегрированный пайплайн** для `agents_4_puzzles` с:
- `Megaminx` hybrid orchestrator,
- LLM generation через `g4f` с явным указанием модели,
- optional provider selection / provider fallback,
- neighbour-model lane,
- optional `search_v3` post-refinement,
- сохранением submission, stats и profiles.

## Что умеет
1. брать проект либо из **ZIP-архива**, либо через **`git clone`**;
2. распаковывать или клонировать проект в `/content`;
3. устанавливать зависимости;
4. подтягивать реальные веса neighbour-model **во время запуска**, без хранения `.pth` в Git;
5. настраивать `g4f`-модель, provider и provider-list;
6. запускать гибридный solver;
7. показывать пути к итоговым артефактам.

> Рекомендуемый режим: **GPU runtime** для ускорения `torch`/neighbour-model. Для `g4f` локальный GPU Colab обычно не участвует, если модель вызывается через удалённого provider.


## Как использовать
1. Откройте notebook в Google Colab.
2. При необходимости переключите runtime на **GPU**.
3. В первой кодовой ячейке выберите один из режимов: `upload_zip`, `drive_zip`, `local_zip_path` или `git_clone`.
4. Для архива по умолчанию используйте **no-LFS** версию, а веса подтягивайте отдельной ячейкой.
5. В ячейке **g4f-конфигурации** укажите модель, например `gpt-4o-mini`, `gpt-4o`, `claude-3.5-sonnet`, `gemini-1.5-pro` или другую поддерживаемую вашим provider конфигурацию.
6. При желании задайте конкретный `G4F_PROVIDER`, либо список `G4F_PROVIDER_LIST` для failover.
7. Запустите ячейки сверху вниз.

Если хотите быстрый sanity-check без LLM, можно отключить `GENERATE_LLM`.


In [ ]:
#@title 1. Источник интегрированного репозитория и подготовка workspace

from pathlib import Path
import os
import shutil
import subprocess
import zipfile

SOURCE_MODE = "git_clone"  # @param ["upload_zip", "drive_zip", "local_zip_path", "git_clone"]
ARCHIVE_PATH = "/content/agents_4_puzzles_megaminx_neighbour_llm_integration_nolfs.zip"  # @param {type:"string"}
GIT_REPO_URL = "https://github.com/visualcomments/agents_4_puzzles.git"  # @param {type:"string"}
GIT_BRANCH = "main"  # @param {type:"string"}
WORKDIR = "/content/megaminx_hybrid_workspace"  # @param {type:"string"}
REPO_SUBDIR_HINT = "integrated_repo"  # @param {type:"string"}
FORCE_CLEAN_WORKDIR = True  # @param {type:"boolean"}

workdir = Path(WORKDIR).resolve()
archive_path = Path(ARCHIVE_PATH).resolve()

def _colab_upload_if_needed(dst: Path) -> Path:
    try:
        from google.colab import files  # type: ignore
    except Exception as exc:
        raise RuntimeError("Эта ветка рассчитана на Google Colab (google.colab.files недоступен).") from exc

    uploaded = files.upload()
    if not uploaded:
        raise RuntimeError("Файл не был загружен.")
    name = next(iter(uploaded))
    src = Path(name).resolve()
    if src != dst:
        dst.parent.mkdir(parents=True, exist_ok=True)
        shutil.move(str(src), str(dst))
    return dst

def _looks_like_repo(p: Path) -> bool:
    return (p / "pipeline_cli.py").exists() and (p / "competitions" / "cayley-py-megaminx").exists()

if FORCE_CLEAN_WORKDIR and workdir.exists():
    shutil.rmtree(workdir)

workdir.mkdir(parents=True, exist_ok=True)
repo_dir = None

if SOURCE_MODE == "git_clone":
    repo_dir = workdir / "repo_clone"
    cmd = ["git", "clone", "--depth", "1", "--branch", GIT_BRANCH, GIT_REPO_URL, str(repo_dir)]
    print("[cmd]", " ".join(cmd))
    subprocess.run(cmd, check=True)
    if not _looks_like_repo(repo_dir):
        raise RuntimeError(f"Клонированный репозиторий не похож на ожидаемый layout: {repo_dir}")
else:
    if SOURCE_MODE == "upload_zip":
        if not archive_path.exists():
            print(f"[info] archive not found at {archive_path}; открою upload dialog...")
            archive_path = _colab_upload_if_needed(archive_path)
    elif SOURCE_MODE == "drive_zip":
        try:
            from google.colab import drive  # type: ignore
            drive.mount("/content/drive", force_remount=False)
        except Exception as exc:
            raise RuntimeError("Не удалось смонтировать Google Drive.") from exc
        if not archive_path.exists():
            raise FileNotFoundError(f"ZIP не найден: {archive_path}")
    else:
        if not archive_path.exists():
            raise FileNotFoundError(f"ZIP не найден: {archive_path}")

    extract_dir = workdir / "repo_extract"
    extract_dir.mkdir(parents=True, exist_ok=True)

    with zipfile.ZipFile(archive_path, "r") as zf:
        zf.extractall(extract_dir)

    candidate_roots = []
    hinted = extract_dir / REPO_SUBDIR_HINT
    if hinted.exists():
        candidate_roots.append(hinted)
    candidate_roots.extend([p for p in extract_dir.iterdir() if p.is_dir()])

    for p in candidate_roots:
        if _looks_like_repo(p):
            repo_dir = p.resolve()
            break

    if repo_dir is None:
        raise RuntimeError(
            "Не удалось найти корень интегрированного репозитория после распаковки. "
            f"Проверил: {[str(p) for p in candidate_roots]}"
        )

os.chdir(repo_dir)
print("repo_dir =", repo_dir)
if SOURCE_MODE != "git_clone":
    print("archive_path =", archive_path)
else:
    print("git_repo =", GIT_REPO_URL)
    print("git_branch =", GIT_BRANCH)
print("pwd =", Path.cwd())


In [ ]:
#@title 2. Установка зависимостей

import importlib.util
import subprocess
import sys
from pathlib import Path

repo_dir = Path.cwd().resolve()

EXTRA_PIP_PACKAGES = ""  # @param {type:"string"}
INSTALL_TORCH_IF_MISSING = True  # @param {type:"boolean"}
PIN_TORCH_VERSION = ""  # @param {type:"string"}

def run(cmd):
    print("[cmd]", " ".join(cmd))
    subprocess.run(cmd, check=True)

run([sys.executable, "-m", "pip", "install", "-U", "pip", "setuptools", "wheel"])
run([sys.executable, "-m", "pip", "install", "-r", str(repo_dir / "requirements-full.txt")])

if INSTALL_TORCH_IF_MISSING and importlib.util.find_spec("torch") is None:
    spec = ["torch"]
    if PIN_TORCH_VERSION.strip():
        spec = [f"torch=={PIN_TORCH_VERSION.strip()}"]
    run([sys.executable, "-m", "pip", "install", *spec])

if EXTRA_PIP_PACKAGES.strip():
    run([sys.executable, "-m", "pip", "install", *EXTRA_PIP_PACKAGES.split()])

print("[ok] dependencies installed")


In [ ]:
#@title 3. Секреты и ключи LLM

import os

OPENAI_API_KEY = ""  # @param {type:"string"}
OPENROUTER_API_KEY = ""  # @param {type:"string"}
GEMINI_API_KEY = ""  # @param {type:"string"}
ANTHROPIC_API_KEY = ""  # @param {type:"string"}
GROQ_API_KEY = ""  # @param {type:"string"}
TOGETHER_API_KEY = ""  # @param {type:"string"}
G4F_API_KEY = ""  # @param {type:"string"}

pairs = {
    "OPENAI_API_KEY": OPENAI_API_KEY,
    "OPENROUTER_API_KEY": OPENROUTER_API_KEY,
    "GEMINI_API_KEY": GEMINI_API_KEY,
    "ANTHROPIC_API_KEY": ANTHROPIC_API_KEY,
    "GROQ_API_KEY": GROQ_API_KEY,
    "TOGETHER_API_KEY": TOGETHER_API_KEY,
    "G4F_API_KEY": G4F_API_KEY,
}

set_keys = []
for key, value in pairs.items():
    value = value.strip()
    if value:
        os.environ[key] = value
        set_keys.append(key)

print("Установлены ключи:", set_keys if set_keys else "нет")


In [ ]:
#@title 4. g4f: модель, provider, fallback и discovery

import os
import subprocess
import sys

USE_G4F_FOR_LLM = True  # @param {type:"boolean"}
G4F_MODEL = "r1-1776"  # @param {type:"string"}
G4F_EXTRA_MODELS = ""  # @param {type:"string"}
G4F_PROVIDER = ""  # @param {type:"string"}
G4F_PROVIDER_LIST = ""  # @param {type:"string"}
G4F_ALLOW_AUTO_FALLBACK = True  # @param {type:"boolean"}
AGENTLAB_G4F_USE_ASYNC = True  # @param {type:"boolean"}
AGENTLAB_G4F_ASYNC_STREAM = True  # @param {type:"boolean"}
G4F_BACKEND_API_URL = ""  # @param {type:"string"}

RUN_G4F_DISCOVERY = False  # @param {type:"boolean"}
G4F_DISCOVERY_MODELS = ""  # @param {type:"string"}
G4F_DISCOVERY_TIMEOUT = 15  # @param {type:"integer"}
G4F_DISCOVERY_CONCURRENCY = 5  # @param {type:"integer"}
G4F_DISCOVERY_MAX_MODELS = 20  # @param {type:"integer"}

def _set_or_unset_env(name: str, value: str):
    value = str(value or "").strip()
    if value:
        os.environ[name] = value
    else:
        os.environ.pop(name, None)

_set_or_unset_env("G4F_PROVIDER", G4F_PROVIDER)
_set_or_unset_env("AGENTLAB_G4F_PROVIDER_LIST", G4F_PROVIDER_LIST)
os.environ["AGENTLAB_G4F_PROVIDER_ALLOW_AUTO_FALLBACK"] = "1" if G4F_ALLOW_AUTO_FALLBACK else "0"
os.environ["AGENTLAB_G4F_USE_ASYNC"] = "1" if AGENTLAB_G4F_USE_ASYNC else "0"
os.environ["AGENTLAB_G4F_ASYNC_STREAM"] = "1" if AGENTLAB_G4F_ASYNC_STREAM else "0"
_set_or_unset_env("G4F_BACKEND_API_URL", G4F_BACKEND_API_URL)

def _norm_g4f_model_list(primary: str, extra: str) -> str:
    items = []
    for raw in [primary] + str(extra or "").replace(";", ",").split(","):
        item = str(raw).strip()
        if not item:
            continue
        if not item.lower().startswith("g4f:"):
            item = f"g4f:{item}"
        if item not in items:
            items.append(item)
    return ",".join(items)

EFFECTIVE_G4F_MODELS = _norm_g4f_model_list(G4F_MODEL, G4F_EXTRA_MODELS)
print("EFFECTIVE_G4F_MODELS =", EFFECTIVE_G4F_MODELS)
print("G4F_PROVIDER =", os.environ.get("G4F_PROVIDER", ""))
print("AGENTLAB_G4F_PROVIDER_LIST =", os.environ.get("AGENTLAB_G4F_PROVIDER_LIST", ""))
print("AGENTLAB_G4F_PROVIDER_ALLOW_AUTO_FALLBACK =", os.environ.get("AGENTLAB_G4F_PROVIDER_ALLOW_AUTO_FALLBACK", ""))

if RUN_G4F_DISCOVERY:
    cmd = [
        sys.executable, "pipeline_cli.py", "check-g4f-models",
        "--timeout", str(G4F_DISCOVERY_TIMEOUT),
        "--concurrency", str(G4F_DISCOVERY_CONCURRENCY),
        "--max-models", str(G4F_DISCOVERY_MAX_MODELS),
    ]
    if G4F_DISCOVERY_MODELS.strip():
        cmd.extend(["--models", G4F_DISCOVERY_MODELS.strip()])
    if G4F_PROVIDER.strip():
        cmd.extend(["--provider", G4F_PROVIDER.strip()])
    if G4F_BACKEND_API_URL.strip():
        cmd.extend(["--backend-api-url", G4F_BACKEND_API_URL.strip()])
    print("[cmd]", " ".join(cmd))
    subprocess.run(cmd, check=False)
else:
    print("Discovery пропущен. Включите RUN_G4F_DISCOVERY=True, если хотите проверить модели/provider.")


In [ ]:
#@title 5. Подтянуть реальные веса neighbour-model

import os
import subprocess
import sys
from pathlib import Path

repo_dir = Path.cwd().resolve()

FETCH_NEIGHBOUR_WEIGHTS = True  # @param {type:"boolean"}
ALLOW_FAILURE = True  # @param {type:"boolean"}

weights_dir = repo_dir / "tp" / "cayleypy-neighbour-model-training-main" / "weights"
print("weights_dir =", weights_dir)
print("[info] Веса не должны храниться в Git-репозитории; notebook догружает их отдельно во время запуска.")

if FETCH_NEIGHBOUR_WEIGHTS:
    cmd = [sys.executable, "scripts/fetch_megaminx_neighbour_weights.py"]
    print("[cmd]", " ".join(cmd))
    proc = subprocess.run(cmd, check=False, text=True)
    if proc.returncode != 0 and not ALLOW_FAILURE:
        raise RuntimeError(f"fetch_megaminx_neighbour_weights.py failed with code {proc.returncode}")
else:
    print("Пропущено.")

if weights_dir.exists():
    for p in sorted(weights_dir.glob("*.pth")):
        try:
            size_mb = p.stat().st_size / (1024 * 1024)
            print(f"{p.name}: {size_mb:.1f} MB")
        except FileNotFoundError:
            pass


In [ ]:
#@title 6. Параметры гибридного запуска Megaminx

import os
from pathlib import Path

repo_dir = Path.cwd().resolve()

GENERATE_LLM = True  # @param {type:"boolean"}
LLM_VARIANTS = "structured,heuristic_boosted,master_hybrid,neighbour_model_hybrid"  # @param {type:"string"}

MANUAL_MODELS = ""  # @param {type:"string"}
MANUAL_AGENT_MODELS = ""  # @param {type:"string"}
MANUAL_PLANNER_MODELS = ""  # @param {type:"string"}
MANUAL_CODER_MODELS = ""  # @param {type:"string"}
MANUAL_FIXER_MODELS = ""  # @param {type:"string"}

KEEP_IMPROVING_LLM = True  # @param {type:"boolean"}
LLM_IMPROVEMENT_ROUNDS = 4  # @param {type:"integer"}
MAX_ITERS = 8  # @param {type:"integer"}
BASELINE_PATCH_MAX_ITERS = 4  # @param {type:"integer"}
G4F_RECOVERY_MAX_ITERS = 3  # @param {type:"integer"}

# Эти опции будут добавлены в команду только если текущий `pipeline_cli.py run -h`
# действительно показывает соответствующие флаги. Это защищает Colab от ошибок
# вида: `error: unrecognized arguments: --baseline-patch-max-iters`.

GENERATE_NEIGHBOUR_MODEL = True  # @param {type:"boolean"}
NEIGHBOUR_MODEL_DEVICE = "auto"  # @param ["auto", "cpu", "cuda"]
NEIGHBOUR_MODEL_MAX_ROWS = 32  # @param {type:"integer"}
NEIGHBOUR_MODEL_NUM_STEPS = 18  # @param {type:"integer"}
NEIGHBOUR_MODEL_NUM_ATTEMPTS = 4  # @param {type:"integer"}
NEIGHBOUR_MODEL_BEAM_WIDTH = 128  # @param {type:"integer"}
NEIGHBOUR_MODEL_EVAL_BATCH_SIZE = 2048  # @param {type:"integer"}

RUN_SEARCH_V3 = True  # @param {type:"boolean"}
SKIP_POST_REFINE = False  # @param {type:"boolean"}

LIGHT_MIN_PATH_LEN = 560  # @param {type:"integer"}
AGGRESSIVE_MIN_PATH_LEN = 700  # @param {type:"integer"}
FORCE_AGGRESSIVE_TOP_N = 24  # @param {type:"integer"}

OUTPUT_BASENAME = "submission_cayleypy_llm_neighbour_hybrid_g4f_colab"  # @param {type:"string"}

try:
    import torch  # type: ignore
    auto_device = "cuda" if torch.cuda.is_available() else "cpu"
except Exception:
    auto_device = "cpu"

resolved_device = auto_device if NEIGHBOUR_MODEL_DEVICE == "auto" else NEIGHBOUR_MODEL_DEVICE

def _coalesce_models(manual_value: str, g4f_value: str) -> str:
    manual_value = str(manual_value or "").strip()
    if manual_value:
        return manual_value
    if USE_G4F_FOR_LLM:
        return str(g4f_value or "").strip()
    return ""

MODELS = _coalesce_models(MANUAL_MODELS, EFFECTIVE_G4F_MODELS)
AGENT_MODELS = _coalesce_models(MANUAL_AGENT_MODELS, EFFECTIVE_G4F_MODELS)
PLANNER_MODELS = _coalesce_models(MANUAL_PLANNER_MODELS, EFFECTIVE_G4F_MODELS)
CODER_MODELS = _coalesce_models(MANUAL_CODER_MODELS, EFFECTIVE_G4F_MODELS)
FIXER_MODELS = _coalesce_models(MANUAL_FIXER_MODELS, EFFECTIVE_G4F_MODELS)

submissions_dir = repo_dir / "competitions" / "cayley-py-megaminx" / "submissions"
submissions_dir.mkdir(parents=True, exist_ok=True)

RUN_OUT = submissions_dir / f"{OUTPUT_BASENAME}.csv"
RUN_STATS = submissions_dir / f"{OUTPUT_BASENAME}.stats.json"
RUN_PROFILES = submissions_dir / f"{OUTPUT_BASENAME}.profiles.json"
RUN_LOG = submissions_dir / f"{OUTPUT_BASENAME}.run.log"

print("resolved_device =", resolved_device)
print("MODELS =", MODELS)
print("AGENT_MODELS =", AGENT_MODELS)
print("PLANNER_MODELS =", PLANNER_MODELS)
print("CODER_MODELS =", CODER_MODELS)
print("FIXER_MODELS =", FIXER_MODELS)
print("RUN_OUT =", RUN_OUT)
print("RUN_STATS =", RUN_STATS)
print("RUN_PROFILES =", RUN_PROFILES)
print("RUN_LOG =", RUN_LOG)


In [ ]:
#@title 7. Сборка команды запуска

import shlex
import subprocess
import sys
from pathlib import Path

repo_dir = Path.cwd().resolve()

def add_flag(cmd, flag, value):
    if value is None:
        return
    if isinstance(value, bool):
        if value:
            cmd.append(flag)
        return
    value = str(value).strip()
    if value:
        cmd.extend([flag, value])

def detect_pipeline_run_flags(repo_root: Path):
    try:
        proc = subprocess.run(
            [sys.executable, 'pipeline_cli.py', 'run', '-h'],
            cwd=str(repo_root),
            capture_output=True,
            text=True,
            check=False,
        )
        help_text = (proc.stdout or '') + '\n' + (proc.stderr or '')
    except Exception as exc:
        print(f'[warn] Не удалось прочитать pipeline_cli.py run -h: {exc}')
        help_text = ''
    return {
        'baseline_patch_max_iters': '--baseline-patch-max-iters' in help_text,
        'g4f_recovery_max_iters': '--g4f-recovery-max-iters' in help_text,
        'max_iters': '--max-iters' in help_text,
        'keep_improving': '--keep-improving' in help_text,
        'improvement_rounds': '--improvement-rounds' in help_text,
    }

pipeline_run_flags = detect_pipeline_run_flags(repo_dir)
print('pipeline_cli.py run supported flags:', pipeline_run_flags)

cmd = [
    sys.executable,
    'competitions/cayley-py-megaminx/megaminx_cayleypy_llm_hybrid_solver.py',
    '--out', str(RUN_OUT),
    '--stats-out', str(RUN_STATS),
    '--profiles-out', str(RUN_PROFILES),
    '--neighbour-model-device', str(resolved_device),
    '--neighbour-model-max-rows', str(NEIGHBOUR_MODEL_MAX_ROWS),
    '--neighbour-model-num-steps', str(NEIGHBOUR_MODEL_NUM_STEPS),
    '--neighbour-model-num-attempts', str(NEIGHBOUR_MODEL_NUM_ATTEMPTS),
    '--neighbour-model-beam-width', str(NEIGHBOUR_MODEL_BEAM_WIDTH),
    '--neighbour-model-eval-batch-size', str(NEIGHBOUR_MODEL_EVAL_BATCH_SIZE),
    '--light-min-path-len', str(LIGHT_MIN_PATH_LEN),
    '--aggressive-min-path-len', str(AGGRESSIVE_MIN_PATH_LEN),
    '--force-aggressive-top-n', str(FORCE_AGGRESSIVE_TOP_N),
]

if GENERATE_LLM:
    cmd.append('--generate-llm')
    add_flag(cmd, '--llm-variants', LLM_VARIANTS)
    add_flag(cmd, '--models', MODELS)
    add_flag(cmd, '--agent-models', AGENT_MODELS)
    add_flag(cmd, '--planner-models', PLANNER_MODELS)
    add_flag(cmd, '--coder-models', CODER_MODELS)
    add_flag(cmd, '--fixer-models', FIXER_MODELS)
    add_flag(cmd, '--llm-improvement-rounds', LLM_IMPROVEMENT_ROUNDS)
    add_flag(cmd, '--max-iters', MAX_ITERS)

    if pipeline_run_flags['baseline_patch_max_iters']:
        add_flag(cmd, '--baseline-patch-max-iters', BASELINE_PATCH_MAX_ITERS)
    else:
        print('[compat] Пропускаю --baseline-patch-max-iters: текущий pipeline_cli.py run его не поддерживает')

    if pipeline_run_flags['g4f_recovery_max_iters']:
        add_flag(cmd, '--g4f-recovery-max-iters', G4F_RECOVERY_MAX_ITERS)
    else:
        print('[compat] Пропускаю --g4f-recovery-max-iters: текущий pipeline_cli.py run его не поддерживает')

    if KEEP_IMPROVING_LLM:
        cmd.append('--keep-improving-llm')

if GENERATE_NEIGHBOUR_MODEL:
    cmd.append('--generate-neighbour-model')

if RUN_SEARCH_V3:
    cmd.append('--run-search-v3')

if SKIP_POST_REFINE:
    cmd.append('--skip-post-refine')

RUN_CMD = cmd
print(' '.join(shlex.quote(x) for x in RUN_CMD))


In [ ]:
#@title 7a. Совместимость: патч для старых версий hybrid solver

from pathlib import Path
import re
import subprocess
import sys

repo_dir = Path.cwd().resolve()
solver_path = repo_dir / 'competitions' / 'cayley-py-megaminx' / 'megaminx_cayleypy_llm_hybrid_solver.py'

def _pipeline_run_help_text(repo_root: Path) -> str:
    proc = subprocess.run(
        [sys.executable, 'pipeline_cli.py', 'run', '-h'],
        cwd=str(repo_root),
        capture_output=True,
        text=True,
        check=False,
    )
    return (proc.stdout or '') + '\n' + (proc.stderr or '')

help_text = _pipeline_run_help_text(repo_dir)
supports_baseline_patch = '--baseline-patch-max-iters' in help_text
supports_g4f_recovery = '--g4f-recovery-max-iters' in help_text

text = solver_path.read_text(encoding='utf-8')
original = text

if not supports_baseline_patch:
    text = text.replace(
        "    if baseline_patch_max_iters is not None:\n        cmd.extend(['--baseline-patch-max-iters', str(max(1, int(baseline_patch_max_iters)))])\n",
        "    if baseline_patch_max_iters is not None:\n        print('[compat] skip unsupported --baseline-patch-max-iters for pipeline_cli.py run', flush=True)\n",
    )

if not supports_g4f_recovery:
    text = text.replace(
        "    if g4f_recovery_max_iters is not None:\n        cmd.extend(['--g4f-recovery-max-iters', str(max(1, int(g4f_recovery_max_iters)))])\n",
        "    if g4f_recovery_max_iters is not None:\n        print('[compat] skip unsupported --g4f-recovery-max-iters for pipeline_cli.py run', flush=True)\n",
    )

if text != original:
    solver_path.write_text(text, encoding='utf-8')
    print(f'Patched: {solver_path}')
else:
    print('No patch needed')
print('supports_baseline_patch =', supports_baseline_patch)
print('supports_g4f_recovery =', supports_g4f_recovery)


In [ ]:
#@title 9. Запуск пайплайна с live-логом

import os
import subprocess
import sys
from pathlib import Path

repo_dir = Path.cwd().resolve()

if "RUN_CMD" not in globals():
    raise RuntimeError("Сначала выполните ячейку сборки команды.")

with open(RUN_LOG, "w", encoding="utf-8") as log_f:
    process = subprocess.Popen(
        RUN_CMD,
        cwd=str(repo_dir),
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
        universal_newlines=True,
    )
    assert process.stdout is not None
    for line in process.stdout:
        print(line, end="")
        log_f.write(line)
        log_f.flush()
    return_code = process.wait()

print("\n[done] return_code =", return_code)
if return_code != 0:
    raise RuntimeError(f"Pipeline failed with exit code {return_code}. Проверьте RUN_LOG: {RUN_LOG}")


In [ ]:
#@title 10. Просмотр результатов

import json
from pathlib import Path

import pandas as pd

for path in [RUN_OUT, RUN_STATS, RUN_PROFILES, RUN_LOG]:
    print(path, "exists =" , path.exists())

if RUN_STATS.exists():
    print("\n===== STATS =====")
    print(json.dumps(json.loads(RUN_STATS.read_text(encoding="utf-8")), indent=2, ensure_ascii=False)[:20000])

if RUN_PROFILES.exists():
    print("\n===== PROFILES =====")
    print(json.dumps(json.loads(RUN_PROFILES.read_text(encoding="utf-8")), indent=2, ensure_ascii=False)[:20000])

if RUN_OUT.exists():
    print("\n===== SUBMISSION HEAD =====")
    display(pd.read_csv(RUN_OUT).head())

if RUN_LOG.exists():
    print("\n===== LOG TAIL =====")
    tail_lines = RUN_LOG.read_text(encoding="utf-8", errors="ignore").splitlines()[-80:]
    print("\n".join(tail_lines))


In [ ]:
#@title 11. Скачать итоговые артефакты на локальный компьютер

DOWNLOAD_OUTPUTS = True  # @param {type:"boolean"}

if DOWNLOAD_OUTPUTS:
    try:
        from google.colab import files  # type: ignore
    except Exception as exc:
        raise RuntimeError("google.colab.files недоступен") from exc

    for p in [RUN_OUT, RUN_STATS, RUN_PROFILES, RUN_LOG]:
        if p.exists():
            print("downloading", p)
            files.download(str(p))
else:
    print("Пропущено.")


In [ ]:
#@title 12. Kaggle credentials: добавить `kaggle.json` inline

import json
import os
import stat
from pathlib import Path

WRITE_KAGGLE_JSON = False  # @param {type:"boolean"}
KAGGLE_USERNAME = ""  # @param {type:"string"}
KAGGLE_KEY = ""  # @param {type:"string"}
KAGGLE_JSON_INLINE = '{\n  "username": "",\n  "key": ""\n}'  # @param {type:"string"}

kaggle_dir = Path.home() / ".kaggle"
kaggle_path = kaggle_dir / "kaggle.json"

def _build_payload() -> dict:
    username = str(KAGGLE_USERNAME or "").strip()
    key = str(KAGGLE_KEY or "").strip()
    if username and key:
        return {"username": username, "key": key}
    raw = str(KAGGLE_JSON_INLINE or "").strip()
    if not raw:
        raise ValueError("KAGGLE_JSON_INLINE пустой")
    payload = json.loads(raw)
    if not isinstance(payload, dict):
        raise ValueError("kaggle.json должен быть JSON-объектом")
    if not payload.get("username") or not payload.get("key"):
        raise ValueError("В kaggle.json должны быть поля username и key")
    return payload

if WRITE_KAGGLE_JSON:
    payload = _build_payload()
    kaggle_dir.mkdir(parents=True, exist_ok=True)
    kaggle_path.write_text(json.dumps(payload, ensure_ascii=False, indent=2), encoding="utf-8")
    os.chmod(kaggle_path, stat.S_IRUSR | stat.S_IWUSR)
    os.environ["KAGGLE_USERNAME"] = payload["username"]
    os.environ["KAGGLE_KEY"] = payload["key"]
    print(f"Записан {kaggle_path}")
    print("chmod:", oct(kaggle_path.stat().st_mode & 0o777))
    print("username:", payload["username"])
else:
    print("WRITE_KAGGLE_JSON=False — файл не записан")


In [ ]:
#@title 13. Опционально: Kaggle preflight и submit

import os
import shutil
import subprocess
import sys
from pathlib import Path

RUN_KAGGLE_PREFLIGHT = True  # @param {type:"boolean"}
SUBMIT_TO_KAGGLE = True  # @param {type:"boolean"}
KAGGLE_JSON_MODE = "upload"  # @param ["none", "upload", "path"]
KAGGLE_JSON_PATH = ""  # @param {type:"string"}
KAGGLE_MESSAGE = "colab megaminx neighbour+llm hybrid"  # @param {type:"string"}

repo_dir = Path.cwd().resolve()
kaggle_json_path = None

if KAGGLE_JSON_MODE == "upload":
    try:
        from google.colab import files  # type: ignore
    except Exception as exc:
        raise RuntimeError("google.colab.files недоступен") from exc
    uploaded = files.upload()
    if uploaded:
        first = next(iter(uploaded))
        kaggle_json_path = str(Path(first).resolve())
elif KAGGLE_JSON_MODE == "path":
    if KAGGLE_JSON_PATH.strip():
        kaggle_json_path = str(Path(KAGGLE_JSON_PATH).resolve())

if RUN_KAGGLE_PREFLIGHT:
    preflight_cmd = [
        sys.executable, "pipeline_cli.py", "kaggle-preflight",
        "--competition", "cayley-py-megaminx",
    ]
    if kaggle_json_path:
        preflight_cmd.extend(["--kaggle-json", kaggle_json_path])
    print("[cmd]", " ".join(preflight_cmd))
    subprocess.run(preflight_cmd, check=False)

if SUBMIT_TO_KAGGLE:
    if not RUN_OUT.exists():
        raise FileNotFoundError(f"Submission не найдена: {RUN_OUT}")
    submit_cmd = [
        sys.executable,
        "competitions/cayley-py-megaminx/megaminx_cayleypy_llm_hybrid_solver.py",
        "--out", str(RUN_OUT),
        "--stats-out", str(RUN_STATS),
        "--profiles-out", str(RUN_PROFILES),
        "--submit",
        "--message", KAGGLE_MESSAGE,
    ]
    if kaggle_json_path:
        submit_cmd.extend(["--kaggle-json", kaggle_json_path])
    print("[cmd]", " ".join(submit_cmd))
    subprocess.run(submit_cmd, check=False)
else:
    print("Пропущено.")


## Примечания
- В архиве репозитория уже добавлен CLI-флаг `--baseline-patch-max-iters`; compatibility-ячейка остаётся как дополнительная страховка для старых клонов.
- Добавлен compatibility-check: блокнот читает `pipeline_cli.py run -h` и не передаёт неподдерживаемые флаги вроде `--baseline-patch-max-iters`, чтобы не падать на `argparse`.
- Если хотите **быстрый прогон без LLM**, отключите `GENERATE_LLM`.
- Для ZIP по умолчанию используйте **`agents_4_puzzles_megaminx_neighbour_llm_integration_nolfs.zip`**.
- Веса `*.pth` не нужно коммитить в Git: notebook подтягивает их через `scripts/fetch_megaminx_neighbour_weights.py`.
- Теперь можно записать `~/.kaggle/kaggle.json` прямо из notebook: либо через поля `KAGGLE_USERNAME` / `KAGGLE_KEY`, либо вставкой полного JSON в `KAGGLE_JSON_INLINE`.
- Для `g4f` модель обычно удобнее задавать в поле `G4F_MODEL`, а не вручную писать `g4f:` в `MODELS`: notebook сам добавит нужный префикс.
- Если нужен строгий provider selection, задайте `G4F_PROVIDER` или `G4F_PROVIDER_LIST` и выключите `G4F_ALLOW_AUTO_FALLBACK`.
- Если уже починили GitHub-ветку, можно выбрать `SOURCE_MODE="git_clone"`.
- Если запускаете через API-ключи, лучше не сохранять notebook с заполненными секретами.
- Все основные артефакты складываются в `competitions/cayley-py-megaminx/submissions/`.
